In [ ]:
import pandas as pd
import numpy as np
import gender_guesser.detector as gender
import warnings
from tensorflow.keras.utils import get_file
from deepface import DeepFace
import os
import requests

warnings.filterwarnings('ignore')


26-04-23 23:42:07 - Directory C:\Users\lcsama\.deepface has been created
26-04-23 23:42:07 - Directory C:\Users\lcsama\.deepface\weights has been created


In [ ]:
# Load
file_path = r'D:\PythonProjects\Airbnb-Performace\MergedEdinburgh\2024-08-17_listings.csv.gz'
df = pd.read_csv(file_path)

# CV model fix
def fixed_cv_model(data):
    cond1 = data['host_has_profile_pic'] == 1
    cond2 = ~data['host_name'].str.contains('&', na=False)
    cond3 = ~data['host_name'].str.contains('Or ', case=True, na=False)
    cond4 = ~data['host_name'].str.contains('And ', case=True, na=False)
    cond5 = ~data['host_name'].str.contains('AND', case=True, na=False)
    cond6 = ~data['host_name'].str.contains('Randy\\\Zuke', na=False)
    cond7 = ~data['host_name'].str.contains('\+', case=True, na=False)

    cv_data = data[['id', 'host_picture_url']][cond1 & cond2 & cond3 & cond4 & cond5 & cond6 & cond7].copy()

    cv_data = cv_data.reset_index(drop=True)

    cv_data['cv_gender'] = np.nan

    links = cv_data['host_picture_url'].tolist()

    for i in range(len(cv_data)):
        try:
            host_pic = get_file('host' + str(i) + '.jpg', links[i])
            
            result = DeepFace.analyze(host_pic, actions=['gender'], enforce_detection=False)

            if isinstance(result, list): 
                result = result[0]
            
            if 'dominant_gender' in result:
                gender_pred = result['dominant_gender']
            else:
                gender_pred = result['gender']
                
            if isinstance(gender_pred, dict):
                gender_pred = max(gender_pred, key=gender_pred.get)
                
            cv_data.loc[i, 'cv_gender'] = gender_pred
            
        except Exception as e:
            cv_data.loc[i, 'cv_gender'] = 0
            continue
            
    return cv_data

In [ ]:
# Phase 1: NLP 

d = gender.Detector()

def get_first_name(name):
    if pd.isna(name):
        return ""
    return str(name).split()[0]

df['first_name'] = df['host_name'].apply(get_first_name)
df['name_gender'] = df['first_name'].apply(d.get_gender)

couple_pattern = r'(&|Or |And |AND|Randy\\Zuke|\+)'
df['is_couple'] = df['host_name'].astype(str).str.contains(couple_pattern, regex=True, na=False)

print(df[['host_name', 'first_name', 'name_gender', 'is_couple']].head(15))

print(df['name_gender'].value_counts(dropna=False))

print("\n=== Result ===")
print(df['is_couple'].value_counts(dropna=False))

Phase 1: 正在进行 NLP 名字推断...
    host_name first_name name_gender  is_couple
0   Charlotte  Charlotte      female      False
1      Gordon     Gordon        male      False
2       Trish      Trish      female      False
3       Shaun      Shaun        male      False
4        Mark       Mark        male      False
5    Francois   Francois     unknown      False
6     Natalie    Natalie      female      False
7       Susie      Susie      female      False
8        Rona       Rona      female      False
9       Derek      Derek        male      False
10       Alan       Alan        male      False
11    Michele    Michele      female      False
12    Daniela    Daniela      female      False
13      James      James        male      False
14      Simon      Simon        male      False

=== NLP 名字识别结果分布 ===
name_gender
female           2377
male             2244
unknown          1065
mostly_female     156
mostly_male       142
andy              103
Name: count, dtype: int64

=== Couple 或 

In [ ]:
# Phase 2 DeepFace for missing cases
print("Phase 2: Rerunning DeepFace CV model...")

has_pic_mask = df['host_has_profile_pic'].astype(str).str.lower().isin(['t', 'true', '1.0', '1'])
missing_mask = (df['name_gender'].isin(['unknown', 'andy'])) & (~df['is_couple']) & has_pic_mask

missing_cases_df = df[missing_mask].copy()
print(f"Missing cases needed to be identified: {len(missing_cases_df)}")

if len(missing_cases_df) > 0:
    cv_results = fixed_cv_model(missing_cases_df)

    if 'cv_gender' in df.columns:
        df.drop(columns=['cv_gender'], inplace=True)
        
    df = df.merge(cv_results[['id', 'cv_gender']], on='id', how='left')
else:
    df['cv_gender'] = np.nan
    print("Error!")


Phase 2: Rerunning DeepFace CV model...
筛选出需要 CV 识别的样本数量：1092


In [ ]:
# Phase 3: Complete Refactoring Pipeline for CV Image Recognition

# Filter the number of samples requiring recognition

print("=== Filter samples for CV recognition ===")

has_pic_mask = df['host_has_profile_pic'].astype(str).str.lower().isin(['t', 'true', '1.0', '1'])

missing_mask = (df['name_gender'].isin(['unknown', 'andy'])) & (~df['is_couple']) & has_pic_mask

cv_df = df[missing_mask][['id', 'host_picture_url']].copy()
cv_df = cv_df.reset_index(drop=True)
cv_df['cv_gender'] = ""

print(f" Successfully filtered! A total of {len(cv_df)} samples require image recognition.\n")


# Download Sample Images & Identify Gender

print("=== Starting image download and DeepFace recognition ===")

# Create a visible folder to store the downloaded images
save_dir = r'D:\PythonProjects\Airbnb-Performace\MergedEdinburgh\host_faces'
os.makedirs(save_dir, exist_ok=True)
print(f" Images will be saved in this folder: {save_dir}\n")

headers = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/115.0.0.0 Safari/537.36'
}

success_count = 0
fail_count = 0

for i in range(len(cv_df)):
    host_id = cv_df.loc[i, 'id']
    img_url = cv_df.loc[i, 'host_picture_url']
    pic_path = os.path.join(save_dir, f"host_{host_id}.jpg")
    
    try:
        if not os.path.exists(pic_path):
            response = requests.get(img_url, headers=headers, stream=True, timeout=10)
            if response.status_code == 200:
                with open(pic_path, 'wb') as f:
                    for chunk in response.iter_content(1024):
                        f.write(chunk)
            else:
                raise ValueError(f"下载被服务器拒绝，HTTP 状态码: {response.status_code}")
        

        result = DeepFace.analyze(img_path=pic_path, actions=['gender'], enforce_detection=False)

        if isinstance(result, list): 
            result = result[0]
            
        if 'dominant_gender' in result:
            gender_pred = result['dominant_gender']
        else:
            gender_pred = max(result['gender'], key=result['gender'].get)
            
        cv_df.loc[i, 'cv_gender'] = gender_pred
        success_count += 1
        
    except Exception as e:
        cv_df.loc[i, 'cv_gender'] = "unknown"
        fail_count += 1
        if fail_count <= 3:
            print(f" ID {host_id} failed: {str(e)[:150]}")

    # Process
    if (i + 1) % 500 == 0 or (i + 1) == len(cv_df):
        print(f" Process: {i+1}/{len(cv_df)} (sucessed: {success_count}, failed: {fail_count})")

print("\n=== Phase 2 Completed! ===")
print(f"Final Gender Successfully Extracted: {success_count}")
print(f"Unidentified (Image Invalid/Landscape Photo): {fail_count}")


In [ ]:
print(cv_df['cv_gender'].value_counts(dropna=False))

if 'cv_gender' in df.columns:
    df.drop(columns=['cv_gender'], inplace=True)

df['id'] = df['id'].astype(int)
cv_df['id'] = cv_df['id'].astype(int)
df = df.merge(cv_df[['id', 'cv_gender']], on='id', how='left')

print("\nResult：")
print(df['cv_gender'].value_counts(dropna=False))

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

sns.set_theme(style="whitegrid")
plt.rcParams['font.sans-serif'] = ['SimHei']
plt.rcParams['axes.unicode_minus'] = False

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# 1: NLP
sns.countplot(data=df, x='name_gender', ax=axes[0], order=df['name_gender'].value_counts().index, palette='viridis')
axes[0].set_title('Phase 1: NLP Name Gender', fontsize=12)

# 2: CV
cv_data = df[df['cv_gender'].notna()]
if len(cv_data) > 0:
    sns.countplot(data=cv_data, x='cv_gender', ax=axes[1], order=cv_data['cv_gender'].value_counts().index, palette='Set2')
axes[1].set_title('Phase 2: DeepFace CV Distribution', fontsize=12)

# 3
total_hosts = len(df)
couple_count = df['is_couple'].sum()
nlp_success_count = ((~df['is_couple']) & (df['name_gender'].isin(['male', 'mostly_male', 'female', 'mostly_female']))).sum()

cv_success_mask = df['cv_gender'].astype(str).str.lower().isin(['man', 'woman', 'male', 'female'])
cv_success_count = cv_success_mask.sum() 

cv_failed_count = (df['cv_gender'] == 0).sum() 
remaining_unknown = total_hosts - couple_count - nlp_success_count - cv_success_count - cv_failed_count

labels = ['NLP Identified', 'Identified as Couple', 'CV Success', 'CV Failed', 'Still Unknown']
sizes = [nlp_success_count, couple_count, cv_success_count, cv_failed_count, remaining_unknown]
colors = ['#66b3ff', '#99ff99', '#ffb3e6', '#c2c2f0', '#ff9999']

filtered_sizes = [s for s in sizes if s > 0]
filtered_labels = [l for s, l in zip(sizes, labels) if s > 0]
filtered_colors = [c for s, c in zip(sizes, colors) if s > 0]

axes[2].pie(filtered_sizes, labels=filtered_labels, autopct='%1.1f%%', startangle=140, colors=filtered_colors)
axes[2].set_title('Overall Gender Detection Coverage', fontsize=12)

plt.tight_layout()
plt.show()

print(f"=== Statistics Overview ===")
print(f"Total Number of Hosts: {total_hosts}")
print(f"NLP Successful Identifications: {nlp_success_count}")
print(f"Directly Classified as Couple: {couple_count}")
print(f"CV Successful Identifications (Man/Woman): {cv_success_count}")
print(f"CV Identification Failed (e.g., Landscape Images): {cv_failed_count}")
print(f"Remaining for Imputation (No Avatar/Unidentifiable): {remaining_unknown}")


In [ ]:
# Phase 4: c, and Categorical conversion
print("Phase 4: Cleaning, merging, and Categorical conversion")

def assign_final_gender(row):
    if row['is_couple']: return 0
    if row['name_gender'] in ['female', 'mostly_female']: return 0
    if row['name_gender'] in ['male', 'mostly_male']: return 1
    if pd.notna(row['cv_gender']) and row['cv_gender'] != 0:
        val = str(row['cv_gender']).lower()
        if 'woman' in val or 'female' in val: return 0
        if 'man' in val or 'male' in val: return 1
    return np.nan

df['gender_raw'] = df.apply(assign_final_gender, axis=1)

# Impute demographic groups
mode_gender = df['gender_raw'].mode()[0]
df['gender_final'] = df['gender_raw'].fillna(mode_gender)

df['gender_final'] = df['gender_final'].astype('category')

cols_to_drop = ['first_name', 'name_gender', 'is_couple', 'cv_gender', 'gender_raw']
df.drop(columns=[c for c in cols_to_drop if c in df.columns], inplace=True)

print("\n  gender_final distribution:")
print(df['gender_final'].value_counts(dropna=False))

# Save result
output_path = r'D:\PythonProjects\Airbnb-Performace\202408_listings_with_gender.csv'
df.to_csv(output_path, index=False)
print(f"Saved as: {output_path}")

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

file_path = r'D:\PythonProjects\Airbnb-Performace\MergedEdinburgh\2024-09-13_listings_with_gender_FINAL.csv'
df_final = pd.read_csv(file_path)

sns.set_theme(style="whitegrid")
plt.rcParams['font.sans-serif'] = ['SimHei']
plt.rcParams['axes.unicode_minus'] = False

df_final['gender_label'] = df_final['gender_final'].map({
    0.0: '0 (Female / Couple)', 
    1.0: '1 (Male)'
})

#Visualization
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Count Plot
sns.countplot(
    data=df_final, 
    x='gender_label', 
    ax=axes[0], 
    palette='Set2',
    order=['0 (Female / Couple)', '1 (Male)']
)
axes[0].set_title('Final Host Gender Distribution (Count)', fontsize=14)
axes[0].set_xlabel('Gender Category', fontsize=12)
axes[0].set_ylabel('Number of Hosts', fontsize=12)

for p in axes[0].patches:
    axes[0].annotate(f'{int(p.get_height())}', 
                     (p.get_x() + p.get_width() / 2., p.get_height()), 
                     ha='center', va='bottom', 
                     fontsize=12, color='black', xytext=(0, 5), 
                     textcoords='offset points')

# Pie Chart
gender_counts = df_final['gender_label'].value_counts()
axes[1].pie(
    gender_counts, 
    labels=gender_counts.index, 
    autopct='%1.1f%%', 
    startangle=90, 
    colors=sns.color_palette('Set2'), 
    wedgeprops={'edgecolor': 'white', 'linewidth': 1.5},
    textprops={'fontsize': 12}
)
axes[1].set_title('Final Host Gender Proportion', fontsize=14)

plt.suptitle('Edinburgh Airbnb Hosts Demographic (Gender)', fontsize=16, fontweight='bold', y=1.05)
plt.tight_layout()
plt.show()

print("=== Final Demographic (Gender) Statistics Overview ===")
print(f"Total Processed Hosts: {len(df_final)}")
print("\nSpecific Category Counts:")
print(df_final['gender_final'].value_counts(dropna=False).rename(index={0.0: '0 (Female/Couple/Imputed)', 1.0: '1 (Male)'}))